# Pedigree Selection Tutorial - Python Version

This notebook replicates the AlphaSimR pedigree selection line breeding tutorial using **AlphaSimPy**.
It demonstrates a phenotypic line breeding program with **pedigree-based family selection** across multiple generations and yield trial stages.

**Authors**: Translated from AlphaSimR tutorial by Jon Bancic, Philip Greenspoon, Chris Gaynor, Gregor Gorjanc  
**Python translation**: AlphaSimPy version  
**Package**: AlphaSimPy


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from AlphaSimPy import (
    run_macs,
    SimParam,
    new_pop,
    rand_cross,
    self as self_pop,
    set_pheno,
    select_ind,
    mean_g,
    var_g,
    meanP,
    merge_pops,
    nInd,
)

print("AlphaSimPy Pedigree Selection Tutorial")
print("All libraries imported successfully!")

## Global Parameters

Set up the simulation parameters for the pedigree selection breeding program (mirroring the AlphaSimR PedigreeSelection example).

In [ ]:
# Number of simulation replications and breeding cycles
n_reps = 1          # nReps
n_burnin = 20       # nBurnin
n_future = 20       # nFuture
n_cycles = n_burnin + n_future

# Genome simulation
n_qtl = 1000        # nQtl per chromosome
n_snp = 0           # nSnp per chromosome
n_chr = 1           # nChr

# Initial parents mean and variance
init_mean_g = 1.0   # initMeanG
init_var_g = 1.0    # initVarG
init_var_env = 1e-6 # initVarEnv (env variance for traits)
init_var_gxe = 2.0  # initVarGE (GxE variance)
var_e = 4.0         # varE (yield trial error variance)

# Breeding program details
n_parents = 20      # nParents
n_crosses = 40      # nCrosses

# Number of progeny per selfed individual in each stage
n_F2 = 100          # nF2
plants_per_row = 20 # plantsPerRow

# Number of individuals to select in each stage
n_PYT = 100         # nPYT
n_AYT = 50          # nAYT
n_EYT = 10          # nEYT
n_sel_F2 = 10       # nSelF2
n_sel_F3 = 4        # nSelF3
n_sel_F4 = 4        # nSelF4
n_sel_F5 = 4        # nSelF5

# Number of rows to select in each stage
n_row_F3 = 10       # nRowF3
n_row_F4 = 10       # nRowF4
n_row_F5 = 10       # nRowF5
n_row_F6 = 4        # nRowF6

# Effective replication of yield trials (controls h2 at each stage)
rep_F6  = 4.0 / 9.0 # repF6  (approx h2 = 0.1)
rep_PYT = 1.0       # repPYT (approx h2 = 0.2)
rep_AYT = 4.0       # repAYT (approx h2 = 0.5)
rep_EYT = 8.0       # repEYT (approx h2 = 0.7)

print("Simulation parameters initialised.")

## Create Founders and Initial Parents

This mirrors `CreateParents.R`: simulate founder haplotypes, set up `SimParam`, define a trait with GxE, enable pedigree tracking, create founder parents, and assign initial EYT-like phenotypes.

In [ ]:
print("Creating founders and initial parents...")

# Generate initial haplotypes (founders)
founder_pop = run_macs(
    n_ind=n_parents,
    n_chr=n_chr,
    seg_sites=n_qtl + n_snp,
    inbred=True,
    species="WHEAT",
)

# Simulation parameters
SP = SimParam(founder_pop)

# (Optional) SNP chip restriction as in R (we skip restrSegSites for simplicity)
if n_snp > 0:
    SP.restrSegSites(nQtlPerChr=n_qtl, minSnpPerChr=n_snp)
    SP.addSnpChip(n_snp)

# Add trait with GxE and small environmental variance (initVarEnv, initVarGE)
SP.addTraitAG(
    nQtlPerChr=n_qtl,
    mean=init_mean_g,
    var=init_var_g,
    varEnv=init_var_env,
    var_gxE=init_var_gxe,
)

# Enable pedigree tracking
SP.setTrackPed(True)

# Create founder parents
Parents = new_pop(founder_pop, sim_param=SP)

# Add phenotype reflecting evaluation in EYT
Parents = set_pheno(Parents, var_e=var_e, reps=rep_EYT, sim_param=SP)

print(f"✓ Founders created: {founder_pop.n_ind} individuals, {founder_pop.n_loci[0]} loci")
print(f"✓ Parents created: {Parents.n_ind} individuals, {Parents.n_traits} traits")
print(f"  Mean G: {mean_g(Parents)[0]:.3f}, Var G: {var_g(Parents)[0]:.3f}")

## Family-Level Accuracy Helper

This reproduces `accuracy_family()` from `ExtraFunctions.R`, computing the correlation between family mean phenotypes and family mean genetic values.

In [ ]:
def accuracy_family(pop):
    """Compute between-family accuracy (corr of family mean G vs family mean P).

    This mirrors the R function accuracy_family() in ExtraFunctions.R.
    """
    mothers = np.array(pop.mother)
    fathers = np.array(pop.father)
    families = np.unique(np.stack([mothers, fathers], axis=1), axis=0)

    family_pops = []
    for mother_id, father_id in families:
        mask = (mothers == mother_id) & (fathers == father_id)
        # Build a subpopulation of this family by index
        idx = np.where(mask)[0].tolist()
        if not idx:
            continue
        family = select_ind(pop, n_ind=len(idx), candidates=idx, sim_param=SP)
        family_pops.append(family)

    if not family_pops:
        return np.nan

    phenotypes = np.array([meanP(f)[0] for f in family_pops])
    gvs = np.array([mean_g(f)[0] for f in family_pops])

    if phenotypes.size < 2:
        return np.nan

    return float(np.corrcoef(gvs, phenotypes)[0, 1])

print("Family accuracy helper defined.")

## Fill Breeding Pipeline with Pedigree Selection

This cell mirrors `FillPipeline.R`. It:

- Creates F1 crosses from the current `Parents`.
- Selfs and selects within F2–F5 rows using within-family row means (`meanP`).
- Derives F6 lines and evaluates them in replicated yield trials.
- Progresses to PYT, AYT, and EYT stages.

For simplicity, we run the pipeline once to steady-state (cohort 1–9), as in the R script.

## Advance Year Function

This function mirrors `AdvanceYear.R`. It advances the breeding pipeline by one year, working backwards through stages (EYT → AYT → PYT → F6 → F5 → F4 → F3 → F2 → F1). This is called each year during burn-in and future phases.

In [ ]:
def advance_year(Parents, F1, F2, F3, F4, F5, F6, PYT, AYT, EYT, year, output):
    """
    Advance breeding program by 1 year.
    
    Works backwards through pipeline to avoid copying data.
    Mirrors AdvanceYear.R logic.
    """
    # Stage 9: EYT
    EYT = select_ind(AYT, n_EYT, sim_param=SP)
    EYT = set_pheno(EYT, var_e=var_e, reps=rep_EYT, sim_param=SP)
    
    # Stage 8: AYT
    AYT = select_ind(PYT, n_AYT, sim_param=SP)
    AYT = set_pheno(AYT, var_e=var_e, reps=rep_AYT, sim_param=SP)
    
    # Stage 7: PYT
    output['accSel'][year-1] = accuracy_family(F6)
    PYT = select_ind(F6, n_PYT, sim_param=SP)
    PYT = set_pheno(PYT, var_e=var_e, reps=rep_PYT, sim_param=SP)
    
    # Stage 6: F6 lines
    F6_list = []
    for i in range(n_crosses):
        this_F5 = F5[i]
        n_rows = nInd(this_F5)
        F6_lines = []
        F6_pheno = []
        for j in range(n_rows):
            # Derive line from F5 row (no selfing, just select the plant)
            line = select_ind(this_F5, n_ind=1, candidates=[j], sim_param=SP)
            F6_lines.append(line)
            # Evaluate row for selection
            F6_j = self_pop(this_F5, n_progeny=plants_per_row, parents=[j], sim_param=SP)
            F6_pheno.append(meanP(F6_j)[0])
        # Select top n_row_F6 rows per cross
        take = np.argsort(F6_pheno)[::-1][:n_row_F6]
        F6_lines = [F6_lines[k] for k in take]
        F6_list.append(merge_pops(F6_lines))
    F6 = merge_pops(F6_list)
    F6 = set_pheno(F6, var_e=var_e, reps=rep_F6, sim_param=SP)
    
    # Stage 5: F5 rows
    F5 = []
    for i in range(n_crosses):
        this_F4 = F4[i]
        n_rows = nInd(this_F4)
        F5_rows = []
        F5_pheno = []
        for j in range(n_rows):
            row_pop = self_pop(this_F4, n_progeny=plants_per_row, parents=[j], sim_param=SP)
            F5_rows.append(row_pop)
            F5_pheno.append(meanP(row_pop)[0])
        take = np.argsort(F5_pheno)[::-1][:n_row_F5]
        F5_rows = [F5_rows[k] for k in take]
        for j in range(n_row_F5):
            F5_rows[j] = set_pheno(F5_rows[j], var_e=var_e, reps=1, sim_param=SP)
            F5_rows[j] = select_ind(F5_rows[j], n_sel_F5, sim_param=SP)
        F5.append(merge_pops(F5_rows))
    
    # Stage 4: F4 rows
    F4 = []
    for i in range(n_crosses):
        this_F3 = F3[i]
        n_rows = nInd(this_F3)
        F4_rows = []
        F4_pheno = []
        for j in range(n_rows):
            row_pop = self_pop(this_F3, n_progeny=plants_per_row, parents=[j], sim_param=SP)
            F4_rows.append(row_pop)
            F4_pheno.append(meanP(row_pop)[0])
        take = np.argsort(F4_pheno)[::-1][:n_row_F4]
        F4_rows = [F4_rows[k] for k in take]
        for j in range(n_row_F4):
            F4_rows[j] = set_pheno(F4_rows[j], var_e=var_e, reps=1, sim_param=SP)
            F4_rows[j] = select_ind(F4_rows[j], n_sel_F4, sim_param=SP)
        F4.append(merge_pops(F4_rows))
    
    # Stage 3: F3 rows
    F3 = []
    for i in range(n_crosses):
        this_F2 = F2[i]
        n_rows = nInd(this_F2)
        F3_rows = []
        F3_pheno = []
        for j in range(n_rows):
            row_pop = self_pop(this_F2, n_progeny=plants_per_row, parents=[j], sim_param=SP)
            F3_rows.append(row_pop)
            F3_pheno.append(meanP(row_pop)[0])
        take = np.argsort(F3_pheno)[::-1][:n_row_F3]
        F3_rows = [F3_rows[k] for k in take]
        for j in range(n_row_F3):
            F3_rows[j] = set_pheno(F3_rows[j], var_e=var_e, reps=1, sim_param=SP)
            F3_rows[j] = select_ind(F3_rows[j], n_sel_F3, sim_param=SP)
        F3.append(merge_pops(F3_rows))
    
    # Stage 2: F2 selfing and selection
    F2 = []
    for i in range(n_crosses):
        # Self the i-th F1 individual (cross i)
        F2_i = self_pop(F1, n_progeny=n_F2, parents=[i], sim_param=SP)
        F2_i = set_pheno(F2_i, var_e=var_e, reps=1, sim_param=SP)
        F2_i = select_ind(F2_i, n_sel_F2, sim_param=SP)
        F2.append(F2_i)
    
    # Stage 1: F1 crosses
    F1 = rand_cross(Parents, n_crosses, sim_param=SP)
    
    return F1, F2, F3, F4, F5, F6, PYT, AYT, EYT

print("Advance year function defined.")

## Update Parents Function

This function mirrors `UpdateParents.R`. It replaces the 10 oldest parents (indices 0-9) with the 10 new parents from the EYT stage, maintaining a constant population size of `n_parents`.

In [ ]:
def update_parents(Parents, EYT):
    """
    Update parents by replacing oldest 10 with new EYT entries.
    
    Mirrors UpdateParents.R: Parents = c(Parents[11:nParents], EYT)
    In Python, we select indices 10 onwards (0-indexed) and merge with EYT.
    """
    # Select parents from index 10 onwards (keeping n_parents - n_EYT individuals)
    keep_indices = list(range(n_EYT, n_parents))
    if len(keep_indices) > 0:
        Parents_keep = select_ind(Parents, n_ind=len(keep_indices), candidates=keep_indices, sim_param=SP)
        # Merge kept parents with new EYT entries
        Parents_new = merge_pops([Parents_keep, EYT])
    else:
        Parents_new = EYT
    
    return Parents_new

print("Update parents function defined.")

## Burn-in Phase

Run the burn-in phase to establish the breeding program. This mirrors the burn-in loop in `00RUNME.R`, tracking mean genetic value, genetic variance, and selection accuracy each year.

In [ ]:
print("Starting burn-in phase...")

# Initialize output tracking (mirrors output data.frame in 00RUNME.R)
output = {
    'year': list(range(1, n_cycles + 1)),
    'rep': [1] * n_cycles,
    'scenario': ['LinePheno_pedigree'] * n_cycles,
    'mean_g': [0.0] * n_cycles,
    'var_g': [0.0] * n_cycles,
    'accSel': [0.0] * n_cycles,
}

# Burn-in phase
for year in range(1, n_burnin + 1):
    print(f"  Working on burn-in year: {year}")
    
    # Update parents (pick new parents from EYT)
    Parents = update_parents(Parents, EYT)
    
    # Advance breeding program by 1 year
    F1, F2, F3, F4, F5, F6, PYT, AYT, EYT = advance_year(
        Parents, F1, F2, F3, F4, F5, F6, PYT, AYT, EYT, year, output
    )
    
    # Report results (using merged F6 as in R)
    output['mean_g'][year-1] = mean_g(F6)[0]
    output['var_g'][year-1] = var_g(F6)[0]
    
    if year % 5 == 0:
        print(f"    Year {year}: Mean G = {output['mean_g'][year-1]:.3f}, "
              f"Var G = {output['var_g'][year-1]:.3f}, "
              f"Acc = {output['accSel'][year-1]:.3f}")

print(f"\\nBurn-in phase completed!")
print(f"  Final mean G: {output['mean_g'][n_burnin-1]:.3f}")
print(f"  Final var G: {output['var_g'][n_burnin-1]:.3f}")
print(f"  Final accuracy: {output['accSel'][n_burnin-1]:.3f}")

## Future Phase

Continue the breeding program in the future phase, tracking the same metrics.

In [ ]:
print("Starting future phase...")

# Future phase
for year in range(n_burnin + 1, n_burnin + n_future + 1):
    print(f"  Working on future year: {year}")
    
    # Update parents (pick new parents from EYT)
    Parents = update_parents(Parents, EYT)
    
    # Advance breeding program by 1 year
    F1, F2, F3, F4, F5, F6, PYT, AYT, EYT = advance_year(
        Parents, F1, F2, F3, F4, F5, F6, PYT, AYT, EYT, year, output
    )
    
    # Report results (using merged F6 as in R)
    output['mean_g'][year-1] = mean_g(F6)[0]
    output['var_g'][year-1] = var_g(F6)[0]
    
    if (year - n_burnin) % 5 == 0:
        print(f"    Year {year}: Mean G = {output['mean_g'][year-1]:.3f}, "
              f"Var G = {output['var_g'][year-1]:.3f}, "
              f"Acc = {output['accSel'][year-1]:.3f}")

print(f"\\nFuture phase completed!")
print(f"  Final mean G: {output['mean_g'][n_cycles-1]:.3f}")
print(f"  Final var G: {output['var_g'][n_cycles-1]:.3f}")
print(f"  Final accuracy: {output['accSel'][n_cycles-1]:.3f}")

## Analyze and Plot Results

This section mirrors `ANALYZERESULTS.R`, creating plots of genetic gain, genetic variance, and selection accuracy over time.

In [ ]:
# Convert output to numpy arrays for easier plotting
years = np.array(output['year'])
mean_g = np.array(output['mean_g'])
var_g = np.array(output['var_g'])
acc_sel = np.array(output['accSel'])

# Create figure with 3 subplots (mirrors ANALYZERESULTS.R)
fig, axes = plt.subplots(3, 1, figsize=(6, 10))
fig.suptitle('Pedigree Selection Breeding Program Results', fontsize=14, fontweight='bold')

# Plot 1: Genetic Gain
axes[0].plot(years, mean_g, 'b-', linewidth=2, label='Mean Genetic Value')
axes[0].axvline(x=n_burnin, color='r', linestyle='--', alpha=0.5, label='Burn-in/Future')
axes[0].set_xlabel('Year')
axes[0].set_ylabel('Yield')
axes[0].set_title('Genetic Gain')
axes[0].grid(True, linestyle='--', alpha=0.3)
axes[0].legend()

# Plot 2: Genetic Variance
axes[1].plot(years, var_g, 'b-', linewidth=2, label='Genetic Variance')
axes[1].axvline(x=n_burnin, color='r', linestyle='--', alpha=0.5, label='Burn-in/Future')
axes[1].set_xlabel('Year')
axes[1].set_ylabel('Variance')
axes[1].set_title('Genetic Variance')
axes[1].grid(True, linestyle='--', alpha=0.3)
axes[1].legend()

# Plot 3: Selection Accuracy
axes[2].plot(years, acc_sel, 'b-', linewidth=2, label='Selection Accuracy')
axes[2].axvline(x=n_burnin, color='r', linestyle='--', alpha=0.5, label='Burn-in/Future')
axes[2].set_xlabel('Year')
axes[2].set_ylabel('Correlation')
axes[2].set_title('Selection Accuracy (Family-level)')
axes[2].grid(True, linestyle='--', alpha=0.3)
axes[2].legend()

plt.tight_layout()
plt.savefig('PedigreeSelection_Results.png', dpi=150, bbox_inches='tight')
print("Results plotted and saved to PedigreeSelection_Results.png")
plt.show()

# Print summary statistics
print("\\nSummary Statistics:")
print(f"  Initial mean G: {mean_g[0]:.3f}")
print(f"  Final mean G: {mean_g[-1]:.3f}")
print(f"  Total genetic gain: {mean_g[-1] - mean_g[0]:.3f}")
print(f"  Initial var G: {var_g[0]:.3f}")
print(f"  Final var G: {var_g[-1]:.3f}")
print(f"  Mean selection accuracy: {np.nanmean(acc_sel):.3f}")

## Summary

This tutorial demonstrated a **pedigree selection** breeding program using AlphaSimPy, which:

1. **Uses family-based selection** across multiple generations (F2–F6) before entering yield trials
2. **Selects based on row means** (`meanP`) within families, maintaining pedigree information
3. **Progresses through yield trial stages** (PYT → AYT → EYT) with increasing replication and heritability
4. **Updates parents** each year by replacing the oldest parents with new elite entries from EYT

Key differences from mass selection:
- **Pedigree tracking** enables family-level selection and accuracy calculations
- **Multi-stage selection** within families (rows) before individual plant selection
- **Longer breeding cycle** (9 stages) compared to simpler mass selection programs

The results show genetic gain over time, changes in genetic variance, and family-level selection accuracy throughout the breeding program.

In [ ]:
print("Filling breeding pipeline with pedigree selection...")

F1 = None
F2 = F3 = F4 = F5 = F6 = None
PYT = AYT = EYT = None

for cohort in range(1, 10):
    print(f"  FillPipeline stage: {cohort} of 9")

    # Stage 1: F1 crosses
    if cohort < 10:
        F1 = rand_cross(Parents, n_crosses, sim_param=SP)

    # Stage 2: F2 selfing and within-cross selection
    if cohort < 9:
        F2 = []  # keep crosses separate
        for i in range(n_crosses):
            F2_i = self_pop(F1, n_progeny=n_F2, parents=[i], sim_param=SP)
            F2_i = set_pheno(F2_i, var_e=var_e, reps=1, sim_param=SP)
            F2_i = select_ind(F2_i, n_sel_F2, sim_param=SP)
            F2.append(F2_i)

    # Stage 3: F3 rows and row means
    if cohort < 8:
        F3 = []
        for i in range(n_crosses):
            this_F2 = F2[i]
            n_rows = nInd(this_F2)
            F3_rows = []
            F3_pheno = []
            for j in range(n_rows):
                row_pop = self_pop(this_F2, n_progeny=plants_per_row, parents=[j], sim_param=SP)
                F3_rows.append(row_pop)
                F3_pheno.append(meanP(row_pop)[0])
            take = np.argsort(F3_pheno)[::-1][:n_row_F3]
            F3_rows = [F3_rows[k] for k in take]
            for j in range(n_row_F3):
                F3_rows[j] = set_pheno(F3_rows[j], var_e=var_e, reps=1, sim_param=SP)
                F3_rows[j] = select_ind(F3_rows[j], n_sel_F3, sim_param=SP)
            F3.append(merge_pops(F3_rows))

    # Stage 4: F4 rows
    if cohort < 7:
        F4 = []
        for i in range(n_crosses):
            this_F3 = F3[i]
            n_rows = nInd(this_F3)
            F4_rows = []
            F4_pheno = []
            for j in range(n_rows):
                row_pop = self_pop(this_F3, n_progeny=plants_per_row, parents=[j], sim_param=SP)
                F4_rows.append(row_pop)
                F4_pheno.append(meanP(row_pop)[0])
            take = np.argsort(F4_pheno)[::-1][:n_row_F4]
            F4_rows = [F4_rows[k] for k in take]
            for j in range(n_row_F4):
                F4_rows[j] = set_pheno(F4_rows[j], var_e=var_e, reps=1, sim_param=SP)
                F4_rows[j] = select_ind(F4_rows[j], n_sel_F4, sim_param=SP)
            F4.append(merge_pops(F4_rows))

    # Stage 5: F5 rows
    if cohort < 6:
        F5 = []
        for i in range(n_crosses):
            this_F4 = F4[i]
            n_rows = nInd(this_F4)
            F5_rows = []
            F5_pheno = []
            for j in range(n_rows):
                row_pop = self_pop(this_F4, n_progeny=plants_per_row, parents=[j], sim_param=SP)
                F5_rows.append(row_pop)
                F5_pheno.append(meanP(row_pop)[0])
            take = np.argsort(F5_pheno)[::-1][:n_row_F5]
            F5_rows = [F5_rows[k] for k in take]
            for j in range(n_row_F5):
                F5_rows[j] = set_pheno(F5_rows[j], var_e=var_e, reps=1, sim_param=SP)
                F5_rows[j] = select_ind(F5_rows[j], n_sel_F5, sim_param=SP)
            F5.append(merge_pops(F5_rows))

    # Stage 6: F6 lines
    if cohort < 5:
        F6_list = []
        for i in range(n_crosses):
            this_F5 = F5[i]
            n_rows = nInd(this_F5)
            F6_lines = []
            F6_pheno = []
            for j in range(n_rows):
                line = select_ind(this_F5, n_ind=1, candidates=[j], sim_param=SP)
                F6_lines.append(line)
                F6_j = self_pop(this_F5, n_progeny=plants_per_row, parents=[j], sim_param=SP)
                F6_pheno.append(meanP(F6_j)[0])
            take = np.argsort(F6_pheno)[::-1][:n_row_F6]
            F6_lines = [F6_lines[k] for k in take]
            F6_list.append(merge_pops(F6_lines))
        F6 = merge_pops(F6_list)
        F6 = set_pheno(F6, var_e=var_e, reps=rep_F6, sim_param=SP)

    # Stage 7: PYT
    if cohort < 4:
        PYT = select_ind(F6, n_PYT, sim_param=SP)
        PYT = set_pheno(PYT, var_e=var_e, reps=rep_PYT, sim_param=SP)

    # Stage 8: AYT
    if cohort < 3:
        AYT = select_ind(PYT, n_AYT, sim_param=SP)
        AYT = set_pheno(AYT, var_e=var_e, reps=rep_AYT, sim_param=SP)

    # Stage 9: EYT
    if cohort < 2:
        EYT = select_ind(AYT, n_EYT, sim_param=SP)
        EYT = set_pheno(EYT, var_e=var_e, reps=rep_EYT, sim_param=SP)

print("Pipeline filled.")
if F6 is not None:
    print(f"  F6 lines: {F6.n_ind} individuals")
if EYT is not None:
    print(f"  EYT entries: {EYT.n_ind} individuals")

  FillPipeline stage: 3 of 9
